In [0]:
from pyspark.sql import functions as F

df = spark.read.table("hive_metastore.gold.gold_features")

df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("load_ratio_c") > 1, 1).otherwise(0)).alias("current_overloads"),
    F.sum(F.when(F.col("load_ratio_v") > 1, 1).otherwise(0)).alias("voltage_overloads"),
    F.sum(F.when((F.col("load_ratio_c") > 1) | (F.col("load_ratio_v") > 1), 1).otherwise(0)).alias("any_overload"),
    F.sum(F.when((F.col("load_ratio_c") > 1) & (F.col("load_ratio_v") > 1), 1).otherwise(0)).alias("both_overload"),
).show()

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("hive_metastore.gold.gold_dataset")

df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("current") > F.col("H_LIM_C"), 1).otherwise(0)).alias("current_above_limit"),
    F.sum(F.when(F.col("voltage") > F.col("H_LIM_V"), 1).otherwise(0)).alias("voltage_above_limit"),
).show()

In [0]:
# =============================================================================
# Thesis Figure: Load Ratio Time Series for a Representative Transformer
# =============================================================================
# Run this in a Databricks notebook cell
# Produces two figures:
#   - Figure 3: Full year view (seasonal patterns + overload rarity)
#   - Figure 4: 2-week zoom around overload events (15-min granularity)
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
SRC_TABLE = "hive_metastore.gold.gold_dataset"  # Use gold_dataset (pre-scaling)
ID_COL = "ID_prefix"
TS_COL = "DATE"

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Find a "representative" transformer
# Criteria: median annual load, has some overload events, complete coverage
# ─────────────────────────────────────────────────────────────────────────────

df = spark.read.table(SRC_TABLE)

# Compute load_ratio if not already present
if "load_ratio_c" not in df.columns:
    df = df.withColumn(
        "load_ratio_c",
        F.when(F.col("H_LIM_C") > 0, F.col("current") / F.col("H_LIM_C")).otherwise(None)
    )

# Aggregate stats per transformer
transformer_stats = (
    df.groupBy(ID_COL)
    .agg(
        F.mean("load_ratio_c").alias("mean_load"),
        F.max("load_ratio_c").alias("max_load"),
        F.sum(F.when(F.col("load_ratio_c") > 1, 1).otherwise(0)).alias("n_overloads"),
        F.count("*").alias("n_rows"),
        F.min(TS_COL).alias("min_date"),
        F.max(TS_COL).alias("max_date"),
    )
    .withColumn("coverage_days", F.datediff(F.col("max_date"), F.col("min_date")))
    .filter(F.col("coverage_days") >= 360)  # At least ~1 year coverage
    .filter(F.col("n_overloads") > 10)       # Has some overloads to show
    .filter(F.col("n_overloads") < 5000)     # Not constantly overloaded
    .orderBy(F.abs(F.col("mean_load") - F.lit(0.5)))  # Closest to median load
)

print("Top 5 candidate transformers:")
transformer_stats.select(
    ID_COL, "mean_load", "max_load", "n_overloads", "n_rows", "coverage_days"
).show(5, truncate=False)

# Select the best candidate
selected_id = transformer_stats.first()[ID_COL]
print(f"\nSelected transformer: {selected_id}")



In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Extract data for the selected transformer
# ─────────────────────────────────────────────────────────────────────────────

df_single = (
    df.filter(F.col(ID_COL) == selected_id)
    .select(TS_COL, "load_ratio_c", "current", "H_LIM_C")
    .orderBy(TS_COL)
    .toPandas()
)

df_single[TS_COL] = pd.to_datetime(df_single[TS_COL])
df_single = df_single.set_index(TS_COL).sort_index()

print(f"Data range: {df_single.index.min()} to {df_single.index.max()}")
print(f"Total rows: {len(df_single):,}")
print(f"Overload events (load_ratio_c > 1): {(df_single['load_ratio_c'] > 1).sum():,}")



In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Find a good 2-week window with overload events
# ─────────────────────────────────────────────────────────────────────────────

# Find dates with overloads
overload_dates = df_single[df_single["load_ratio_c"] > 1].index

# Pick a 2-week window around an overload cluster (prefer summer months)
summer_overloads = overload_dates[(overload_dates.month >= 6) & (overload_dates.month <= 8)]
if len(summer_overloads) > 0:
    center_date = summer_overloads[len(summer_overloads) // 2]
else:
    center_date = overload_dates[len(overload_dates) // 2]

zoom_start = center_date - pd.Timedelta(days=7)
zoom_end = center_date + pd.Timedelta(days=7)

print(f"\nZoom window: {zoom_start.date()} to {zoom_end.date()}")






In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Create Figure 3 — Full Year View
# ─────────────────────────────────────────────────────────────────────────────

fig1, ax1 = plt.subplots(figsize=(12, 4), dpi=150)

# Plot load ratio
ax1.plot(df_single.index, df_single["load_ratio_c"], 
         linewidth=0.3, color="#1f77b4", alpha=0.7, label="Load ratio (current)")

# Threshold line
ax1.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

# Highlight overload points
overload_mask = df_single["load_ratio_c"] > 1
ax1.scatter(df_single.index[overload_mask], df_single.loc[overload_mask, "load_ratio_c"],
            color="#d62728", s=3, alpha=0.6, zorder=5, label="Overload events")

# Formatting
ax1.set_xlabel("Date", fontsize=11)
ax1.set_ylabel("Load Ratio (C / C_limit)", fontsize=11)
ax1.set_ylim(0, min(df_single["load_ratio_c"].max() * 1.1, 3.0))
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title(f"Transformer {selected_id} — Load Ratio Over 12 Months", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig3_full_year.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: /dbfs/tmp/thesis_fig3_full_year.png")


In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Create Figure 4 — 2-Week Zoom
# ─────────────────────────────────────────────────────────────────────────────

df_zoom = df_single.loc[zoom_start:zoom_end]

fig2, ax2 = plt.subplots(figsize=(12, 4), dpi=150)

# Plot load ratio
ax2.plot(df_zoom.index, df_zoom["load_ratio_c"], 
         linewidth=0.8, color="#1f77b4", alpha=0.9, label="Load ratio (current)")

# Threshold line
ax2.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

# Highlight overload points
overload_mask = df_zoom["load_ratio_c"] > 1
if overload_mask.sum() > 0:
    ax2.scatter(df_zoom.index[overload_mask], df_zoom.loc[overload_mask, "load_ratio_c"],
                color="#d62728", s=20, alpha=0.8, zorder=5, label="Overload events")

# Shade overload regions
for idx in df_zoom.index[overload_mask]:
    ax2.axvspan(idx, idx + pd.Timedelta(minutes=15), color="#d62728", alpha=0.1)

# Formatting
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("Load Ratio (C / C_limit)", fontsize=11)
ax2.set_ylim(0, min(df_zoom["load_ratio_c"].max() * 1.1, 2.5))
ax2.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax2.xaxis.set_minor_locator(mdates.HourLocator(interval=12))
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title(f"Transformer {selected_id} — 2-Week Detail ({zoom_start.strftime('%d %b')} – {zoom_end.strftime('%d %b %Y')})", 
              fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig4_zoom.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: /dbfs/tmp/thesis_fig4_zoom.png")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Print summary stats for figure captions
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("SUMMARY FOR FIGURE CAPTIONS")
print("="*60)
print(f"Transformer ID: {selected_id}")
print(f"Data period: {df_single.index.min().strftime('%d %B %Y')} – {df_single.index.max().strftime('%d %B %Y')}")
print(f"Total measurements: {len(df_single):,}")
print(f"Overload events (load_ratio > 1): {(df_single['load_ratio_c'] > 1).sum():,}")
print(f"Overload rate: {100 * (df_single['load_ratio_c'] > 1).mean():.2f}%")
print(f"Mean load ratio: {df_single['load_ratio_c'].mean():.3f}")
print(f"Max load ratio: {df_single['load_ratio_c'].max():.3f}")
print(f"Zoom window: {zoom_start.strftime('%d %B')} – {zoom_end.strftime('%d %B %Y')}")
print(f"Overloads in zoom window: {overload_mask.sum()}")

In [0]:
# =============================================================================
# Thesis Figure: Yearly Average Load Ratio Across All Transformers
# =============================================================================
# Run this in a Databricks notebook cell
# Produces:
#   - Figure: Daily average load_ratio_c and load_ratio_v across all transformers
#   - Shows seasonal patterns at the fleet level
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
SRC_TABLE = "hive_metastore.gold.gold_dataset"  # Use gold_dataset (pre-scaling)
TS_COL = "DATE"

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load data and compute load ratios if needed
# ─────────────────────────────────────────────────────────────────────────────

df = spark.read.table(SRC_TABLE)
# Exclude rows where current or voltage is zero
df = df.filter((F.col("current") > 0) & (F.col("voltage") > 0))
# Compute load_ratio if not already present
if "load_ratio_c" not in df.columns:
    df = df.withColumn(
        "load_ratio_c",
        F.when(F.col("H_LIM_C") > 0, F.col("current") / F.col("H_LIM_C")).otherwise(None)
    )
if "load_ratio_v" not in df.columns:
    df = df.withColumn(
        "load_ratio_v",
        F.when(F.col("H_LIM_V") > 0, F.col("voltage") / F.col("H_LIM_V")).otherwise(None)
    )



In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Aggregate by day — mean, max, and overload rate
# ─────────────────────────────────────────────────────────────────────────────

daily_stats = (
    df.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(
        # Mean load ratios
        F.mean("load_ratio_c").alias("mean_load_ratio_c"),
        F.mean("load_ratio_v").alias("mean_load_ratio_v"),
        # Max load ratios (daily peak)
        F.max("load_ratio_c").alias("max_load_ratio_c"),
        F.max("load_ratio_v").alias("max_load_ratio_v"),
        # Overload rate (% of measurements exceeding threshold)
        F.mean(F.when(F.col("load_ratio_c") > 1, 1).otherwise(0)).alias("overload_rate_c"),
        F.mean(F.when(F.col("load_ratio_v") > 1, 1).otherwise(0)).alias("overload_rate_v"),
        # Count
        F.count("*").alias("n_measurements"),
    )
    .orderBy("date")
    .toPandas()
)

daily_stats["date"] = pd.to_datetime(daily_stats["date"])
daily_stats = daily_stats.set_index("date").sort_index()

print(f"Date range: {daily_stats.index.min().date()} to {daily_stats.index.max().date()}")
print(f"Total days: {len(daily_stats)}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Figure — Mean Load Ratios (Current and Voltage)
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(2, 1, figsize=(12, 7), dpi=150, sharex=True)

# --- Panel A: Load Ratio (Current) ---
ax1 = axes[0]
ax1.plot(daily_stats.index, daily_stats["mean_load_ratio_c"], 
         linewidth=0.8, color="#1f77b4", alpha=0.7, label="Daily mean")

# Add 7-day rolling average for trend
rolling_mean_c = daily_stats["mean_load_ratio_c"].rolling(window=7, center=True).mean()
ax1.plot(daily_stats.index, rolling_mean_c, 
         linewidth=2, color="#1f77b4", label="7-day rolling mean")

# Threshold line
ax1.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

ax1.set_ylabel("Load Ratio (C / C_limit)", fontsize=11)
ax1.set_ylim(0, min(daily_stats["mean_load_ratio_c"].max() * 1.3, 1.2))
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("(a) Current Load Ratio — Daily Average Across All Transformers", fontsize=11, fontweight="bold")

# --- Panel B: Load Ratio (Voltage) ---
ax2 = axes[1]
ax2.plot(daily_stats.index, daily_stats["mean_load_ratio_v"], 
         linewidth=0.8, color="#ff7f0e", alpha=0.7, label="Daily mean")

# Add 7-day rolling average for trend
rolling_mean_v = daily_stats["mean_load_ratio_v"].rolling(window=7, center=True).mean()
ax2.plot(daily_stats.index, rolling_mean_v, 
         linewidth=2, color="#ff7f0e", label="7-day rolling mean")

# Threshold line
ax2.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("Load Ratio (V / V_limit)", fontsize=11)
ax2.set_ylim(0, min(daily_stats["mean_load_ratio_v"].max() * 1.3, 1.2))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title("(b) Voltage Load Ratio — Daily Average Across All Transformers", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig_avg_load_ratios.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: /dbfs/tmp/thesis_fig_avg_load_ratios.png")

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("hive_metastore.gold.gold_dataset")

# Check zeros
df.select(
    F.count("*").alias("total"),
    F.sum(F.when(F.col("current") == 0, 1).otherwise(0)).alias("current_zero"),
    F.sum(F.when(F.col("voltage") == 0, 1).otherwise(0)).alias("voltage_zero"),
    F.sum(F.when((F.col("current") == 0) | (F.col("voltage") == 0), 1).otherwise(0)).alias("either_zero"),
).show()

# Actual overload counts (not averages)
df.select(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("current") > F.col("H_LIM_C"), 1).otherwise(0)).alias("current_above_limit"),
    F.sum(F.when(F.col("voltage") > F.col("H_LIM_V"), 1).otherwise(0)).alias("voltage_above_limit"),
).show()

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Figure — Daily Overload Rate
# ─────────────────────────────────────────────────────────────────────────────

fig2, ax3 = plt.subplots(figsize=(12, 4), dpi=150)

# Plot overload rates (as percentage)
ax3.bar(daily_stats.index, daily_stats["overload_rate_c"] * 100, 
        width=1, color="#1f77b4", alpha=0.6, label="Current overload rate")
ax3.bar(daily_stats.index, daily_stats["overload_rate_v"] * 100, 
        width=1, color="#ff7f0e", alpha=0.6, label="Voltage overload rate")

# Add 7-day rolling average
rolling_rate_c = (daily_stats["overload_rate_c"] * 100).rolling(window=7, center=True).mean()
ax3.plot(daily_stats.index, rolling_rate_c, 
         linewidth=2, color="#1f77b4", label="Current (7-day avg)")

rolling_rate_v = (daily_stats["overload_rate_v"] * 100).rolling(window=7, center=True).mean()
ax3.plot(daily_stats.index, rolling_rate_v, 
         linewidth=2, color="#ff7f0e", label="Voltage (7-day avg)")

ax3.set_xlabel("Date", fontsize=11)
ax3.set_ylabel("Overload Rate (%)", fontsize=11)
ax3.xaxis.set_major_locator(mdates.MonthLocator())
ax3.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax3.legend(loc="upper right", fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.set_title("Daily Overload Rate Across All Transformers", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig_overload_rate.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: /dbfs/tmp/thesis_fig_overload_rate.png")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Print summary statistics
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("SUMMARY STATISTICS FOR FIGURE CAPTIONS")
print("="*60)
print(f"\nDate range: {daily_stats.index.min().strftime('%d %B %Y')} – {daily_stats.index.max().strftime('%d %B %Y')}")
print(f"Total days: {len(daily_stats)}")

print(f"\n--- Current Load Ratio (load_ratio_c) ---")
print(f"Overall mean: {daily_stats['mean_load_ratio_c'].mean():.3f}")
print(f"Min daily mean: {daily_stats['mean_load_ratio_c'].min():.3f}")
print(f"Max daily mean: {daily_stats['mean_load_ratio_c'].max():.3f}")
print(f"Mean overload rate: {daily_stats['overload_rate_c'].mean() * 100:.3f}%")
print(f"Max daily overload rate: {daily_stats['overload_rate_c'].max() * 100:.2f}%")

print(f"\n--- Voltage Load Ratio (load_ratio_v) ---")
print(f"Overall mean: {daily_stats['mean_load_ratio_v'].mean():.3f}")
print(f"Min daily mean: {daily_stats['mean_load_ratio_v'].min():.3f}")
print(f"Max daily mean: {daily_stats['mean_load_ratio_v'].max():.3f}")
print(f"Mean overload rate: {daily_stats['overload_rate_v'].mean() * 100:.3f}%")
print(f"Max daily overload rate: {daily_stats['overload_rate_v'].max() * 100:.2f}%")

# Monthly breakdown
print("\n--- Monthly Overload Rates ---")
monthly = daily_stats.copy()
monthly["month"] = monthly.index.to_period("M")
monthly_stats = monthly.groupby("month").agg({
    "overload_rate_c": "mean",
    "overload_rate_v": "mean"
})
monthly_stats["overload_rate_c"] *= 100
monthly_stats["overload_rate_v"] *= 100
print(monthly_stats.round(3).to_string())

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from pyspark.sql import functions as F
 
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
# Try gold_features first (has events_15m_cnt), fall back to gold_dataset
SRC_TABLE = "hive_metastore.gold.gold_dataset"  # raw values, not scaled
TS_COL = "DATE"
EVENT_COL = "events_15m_cnt"  # The event count feature you engineered
 
# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load data
# ─────────────────────────────────────────────────────────────────────────────
 
df = spark.read.table(SRC_TABLE)
 
# Check if event column exists
if EVENT_COL not in df.columns:
    print(f"Column {EVENT_COL} not found. Available columns:")
    print([c for c in df.columns if "event" in c.lower()])
    raise ValueError(f"Event column {EVENT_COL} not found")
 
print(f"Loaded table with {EVENT_COL} column")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Aggregate by day
# ─────────────────────────────────────────────────────────────────────────────
 
daily_events = (
    df.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(
        F.sum(EVENT_COL).alias("total_events"),
        F.mean(EVENT_COL).alias("mean_events"),
        F.max(EVENT_COL).alias("max_events"),
        F.stddev(EVENT_COL).alias("std_events"),
        # Count rows with at least one event
        F.sum(F.when(F.col(EVENT_COL) > 0, 1).otherwise(0)).alias("rows_with_events"),
        F.count("*").alias("n_measurements"),
    )
    .withColumn("event_rate", F.col("rows_with_events") / F.col("n_measurements") * 100)
    .orderBy("date")
    .toPandas()
)
 
daily_events["date"] = pd.to_datetime(daily_events["date"])
daily_events = daily_events.set_index("date").sort_index()
 
print(f"Date range: {daily_events.index.min().date()} to {daily_events.index.max().date()}")
print(f"Total days: {len(daily_events)}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Figure — Daily Total Events
# ─────────────────────────────────────────────────────────────────────────────
 
fig, axes = plt.subplots(2, 1, figsize=(12, 7), dpi=150, sharex=True)
 
# --- Panel A: Total daily events ---
ax1 = axes[0]
ax1.bar(daily_events.index, daily_events["total_events"], 
        width=1, color="#2ca02c", alpha=0.6, label="Daily total")
 
# Add 7-day rolling average
rolling_total = daily_events["total_events"].rolling(window=7, center=True).mean()
ax1.plot(daily_events.index, rolling_total, 
         linewidth=2, color="#2ca02c", label="7-day rolling mean")
 
ax1.set_ylabel("Total Event Count", fontsize=11)
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("(a) Daily Total SCADA Events Across All Transformers", fontsize=11, fontweight="bold")
 
# --- Panel B: Event rate (% of measurements with events) ---
ax2 = axes[1]
ax2.bar(daily_events.index, daily_events["event_rate"], 
        width=1, color="#9467bd", alpha=0.6, label="Daily rate")
 
# Add 7-day rolling average
rolling_rate = daily_events["event_rate"].rolling(window=7, center=True).mean()
ax2.plot(daily_events.index, rolling_rate, 
         linewidth=2, color="#9467bd", label="7-day rolling mean")
 
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("Event Rate (%)", fontsize=11)
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title("(b) Daily Rate of Measurements with Events (%)", fontsize=11, fontweight="bold")
 
plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig_event_counts.png", dpi=300, bbox_inches="tight")
plt.show()
 
print("\nSaved: /dbfs/tmp/thesis_fig_event_counts.png")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Correlation with overload rate (if you want to show relationship)
# ─────────────────────────────────────────────────────────────────────────────
 
# Load overload data for correlation
df_overload = spark.read.table(SRC_TABLE)
 
# Compute load ratios if needed (check if they exist and are unscaled)
# Note: gold_features may have scaled values, so we check
if "load_ratio_c" in df_overload.columns:
    daily_combined = (
        df_overload.withColumn("date", F.to_date(TS_COL))
        .groupBy("date")
        .agg(
            F.mean(EVENT_COL).alias("mean_events"),
            F.mean(F.when(F.col("load_ratio_c") > 1, 1).otherwise(0)).alias("overload_rate_c"),
            F.mean(F.when(F.col("load_ratio_v") > 1, 1).otherwise(0)).alias("overload_rate_v"),
        )
        .orderBy("date")
        .toPandas()
    )
    
    # Compute correlations
    corr_c = daily_combined["mean_events"].corr(daily_combined["overload_rate_c"])
    corr_v = daily_combined["mean_events"].corr(daily_combined["overload_rate_v"])
    
    print(f"\n--- Correlation: Events vs Overload Rate ---")
    print(f"Events vs Current overload rate: {corr_c:.3f}")
    print(f"Events vs Voltage overload rate: {corr_v:.3f}")

In [0]:
# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: Print summary statistics
# ─────────────────────────────────────────────────────────────────────────────
 
print("\n" + "="*60)
print("SUMMARY STATISTICS FOR FIGURE CAPTIONS")
print("="*60)
print(f"\nDate range: {daily_events.index.min().strftime('%d %B %Y')} – {daily_events.index.max().strftime('%d %B %Y')}")
print(f"Total days: {len(daily_events)}")
 
print(f"\n--- Event Count Statistics ---")
print(f"Overall daily mean (total events): {daily_events['total_events'].mean():,.0f}")
print(f"Min daily total: {daily_events['total_events'].min():,.0f}")
print(f"Max daily total: {daily_events['total_events'].max():,.0f}")
print(f"Mean event rate: {daily_events['event_rate'].mean():.2f}%")
print(f"Max daily event rate: {daily_events['event_rate'].max():.2f}%")
 
# Monthly breakdown
print("\n--- Monthly Event Statistics ---")
monthly = daily_events.copy()
monthly["month"] = monthly.index.to_period("M")
monthly_stats = monthly.groupby("month").agg({
    "total_events": "mean",
    "event_rate": "mean"
})
monthly_stats.columns = ["mean_daily_events", "mean_event_rate"]
print(monthly_stats.round(2).to_string())

In [0]:
df = spark.read.table("hive_metastore.gold.gold_features")


# Quick correlation check: do events correlate with overloads?
df.select(
    F.corr("events_15m_cnt", "label_4h").alias("corr_4h"),
    F.corr("events_15m_cnt", "label_24h").alias("corr_24h"),
    F.corr("events_15m_cnt", "load_ratio_c").alias("corr_load_c"),
    F.corr("events_15m_cnt", "load_ratio_v").alias("corr_load_v")
).show()

In [0]:
# =============================================================================
# Thesis Figure: Weather Variables Throughout the Year
# =============================================================================
# Run this in a Databricks notebook cell
# Produces: 4-panel figure showing daily average weather variables
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from pyspark.sql import functions as F

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
SRC_TABLE = "hive_metastore.gold.gold_dataset"  # or gold_features
TS_COL = "DATE"

# Weather column names (check your actual column names)
TEMP_COL = "temperatura_media_do_ar_horaria_c"
HUMIDITY_COL = "humidade_relativa_media_horaria_percent"
PRECIP_COL = "precipitacao_horaria_mm"
WIND_COL = "velocidade_do_vento_media_horaria_m_per_s"  # or m/s

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load and aggregate by day
# ─────────────────────────────────────────────────────────────────────────────

df = spark.read.table(SRC_TABLE)

# Check column names
print("Available columns with 'temp', 'humid', 'precip', 'vento':")
for c in df.columns:
    if any(x in c.lower() for x in ['temp', 'humid', 'precip', 'vento', 'wind']):
        print(f"  {c}")

# Aggregate by day
daily_weather = (
    df.withColumn("date", F.to_date(TS_COL))
    .withColumn("hour", F.hour(TS_COL))
    .dropDuplicates(["ID_prefix", "date", "hour"])  # one row per transformer-hour
    .groupBy("date")
    .agg(
        F.mean(TEMP_COL).alias("temperature"),
        F.mean(HUMIDITY_COL).alias("humidity"),
        F.mean(PRECIP_COL).alias("precipitation"),  # now sum is correct
        F.mean(WIND_COL).alias("wind_speed"),
    )
    .orderBy("date")
    .toPandas()
)

daily_weather["date"] = pd.to_datetime(daily_weather["date"])
daily_weather = daily_weather.set_index("date").sort_index()

print(f"Date range: {daily_weather.index.min().date()} to {daily_weather.index.max().date()}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Create 4-panel figure
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(4, 1, figsize=(12, 10), dpi=150, sharex=True)

# Color scheme
colors = ["#e41a1c", "#377eb8", "#4daf4a", "#984ea3"]  # red, blue, green, purple

# --- Panel A: Temperature ---
ax1 = axes[0]
ax1.plot(daily_weather.index, daily_weather["temperature"], 
         linewidth=0.8, color=colors[0], alpha=0.6)
rolling_temp = daily_weather["temperature"].rolling(window=7, center=True).mean()
ax1.plot(daily_weather.index, rolling_temp, linewidth=2, color=colors[0], label="7-day avg")
ax1.set_ylabel("Temperature (°C)", fontsize=10)
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("(a) Mean Daily Air Temperature", fontsize=11, fontweight="bold")

# --- Panel B: Humidity ---
ax2 = axes[1]
ax2.plot(daily_weather.index, daily_weather["humidity"], 
         linewidth=0.8, color=colors[1], alpha=0.6)
rolling_humid = daily_weather["humidity"].rolling(window=7, center=True).mean()
ax2.plot(daily_weather.index, rolling_humid, linewidth=2, color=colors[1], label="7-day avg")
ax2.set_ylabel("Humidity (%)", fontsize=10)
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title("(b) Mean Daily Relative Humidity", fontsize=11, fontweight="bold")

# --- Panel C: Precipitation ---
ax3 = axes[2]
ax3.bar(daily_weather.index, daily_weather["precipitation"], 
        width=1, color=colors[2], alpha=0.7)
rolling_precip = daily_weather["precipitation"].rolling(window=7, center=True).mean()
ax3.plot(daily_weather.index, rolling_precip, linewidth=2, color=colors[2], label="7-day avg")
ax3.set_ylabel("Precipitation (mm)", fontsize=10)
ax3.legend(loc="upper right", fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.set_title("(c) Mean Daily Precipitation", fontsize=11, fontweight="bold")

# --- Panel D: Wind Speed ---
ax4 = axes[3]
ax4.plot(daily_weather.index, daily_weather["wind_speed"], 
         linewidth=0.8, color=colors[3], alpha=0.6)
rolling_wind = daily_weather["wind_speed"].rolling(window=7, center=True).mean()
ax4.plot(daily_weather.index, rolling_wind, linewidth=2, color=colors[3], label="7-day avg")
ax4.set_xlabel("Date", fontsize=11)
ax4.set_ylabel("Wind Speed (m/s)", fontsize=10)
ax4.xaxis.set_major_locator(mdates.MonthLocator())
ax4.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax4.legend(loc="upper right", fontsize=9)
ax4.grid(True, alpha=0.3)
ax4.set_title("(d) Mean Daily Wind Speed", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig_weather.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: /dbfs/tmp/thesis_fig_weather.png")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Summary statistics
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*60)
print("SUMMARY STATISTICS FOR FIGURE CAPTION")
print("="*60)
print(f"\nTemperature: {daily_weather['temperature'].mean():.1f}°C mean, "
      f"{daily_weather['temperature'].min():.1f}°C min, "
      f"{daily_weather['temperature'].max():.1f}°C max")
print(f"Humidity: {daily_weather['humidity'].mean():.1f}% mean, "
      f"{daily_weather['humidity'].min():.1f}% min, "
      f"{daily_weather['humidity'].max():.1f}% max")
print(f"Precipitation: {daily_weather['precipitation'].sum():.0f} mm total, "
      f"{daily_weather['precipitation'].max():.1f} mm max daily")
print(f"Wind speed: {daily_weather['wind_speed'].mean():.1f} m/s mean, "
      f"{daily_weather['wind_speed'].max():.1f} m/s max")

# Monthly breakdown
print("\n--- Monthly Averages ---")
monthly = daily_weather.copy()
monthly["month"] = monthly.index.to_period("M")
monthly_avg = monthly.groupby("month").mean()
print(monthly_avg.round(1).to_string())

In [0]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
import pandas as pd

# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
SRC_TABLE = "hive_metastore.gold.gold_dataset"

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load data and compute overloads by season
# ─────────────────────────────────────────────────────────────────────────────

df = spark.read.table(SRC_TABLE)

# Filter out zeros and compute load ratios
df = df.filter((F.col("current") > 0) & (F.col("voltage") > 0))

df = df.withColumn(
    "load_ratio_c",
    F.when(F.col("H_LIM_C") > 0, F.col("current") / F.col("H_LIM_C")).otherwise(None)
).withColumn(
    "load_ratio_v",
    F.when(F.col("H_LIM_V") > 0, F.col("voltage") / F.col("H_LIM_V")).otherwise(None)
)

# Flag overload events
df = df.withColumn(
    "is_overload",
    F.when((F.col("load_ratio_c") > 1) | (F.col("load_ratio_v") > 1), 1).otherwise(0)
)

# Extract month and map to season
df = df.withColumn("month", F.month("DATE"))
df = df.withColumn(
    "season",
    F.when(F.col("month").isin(12, 1, 2), "Winter")
     .when(F.col("month").isin(3, 4, 5), "Spring")
     .when(F.col("month").isin(6, 7, 8), "Summer")
     .otherwise("Autumn")
)

# Aggregate by season
season_stats = (
    df.groupBy("season")
    .agg(
        F.sum("is_overload").alias("overload_count"),
        F.count("*").alias("total_count"),
    )
    .withColumn("overload_rate", F.col("overload_count") / F.col("total_count") * 100)
    .toPandas()
)

# Order seasons correctly
season_order = ["Winter", "Spring", "Summer", "Autumn"]
season_stats["season"] = pd.Categorical(season_stats["season"], categories=season_order, ordered=True)
season_stats = season_stats.sort_values("season")

print(season_stats)

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Pie Chart — Overload Events by Season
# ─────────────────────────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(12, 5), dpi=150)

# Colors for seasons
colors = ["#4A90D9", "#7CB342", "#FFA726", "#8D6E63"]  # Winter=blue, Spring=green, Summer=orange, Autumn=brown

# --- Panel A: Pie chart (count) ---
ax1 = axes[0]
wedges, texts, autotexts = ax1.pie(
    season_stats["overload_count"],
    labels=season_stats["season"],
    autopct=lambda pct: f'{pct:.1f}%\n({int(pct/100 * season_stats["overload_count"].sum()):,})',
    colors=colors,
    explode=[0.02] * 4,
    startangle=90,
    textprops={'fontsize': 10}
)
ax1.set_title("(a) Overload Events by Season", fontsize=11, fontweight="bold")

# --- Panel B: Bar chart (rate) ---
ax2 = axes[1]
bars = ax2.bar(season_stats["season"], season_stats["overload_rate"], color=colors, edgecolor="black", linewidth=0.5)
ax2.set_ylabel("Overload Rate (%)", fontsize=11)
ax2.set_xlabel("Season", fontsize=11)
ax2.set_title("(b) Overload Rate by Season", fontsize=11, fontweight="bold")
ax2.grid(True, alpha=0.3, axis="y")

# Add value labels on bars
for bar, val in zip(bars, season_stats["overload_rate"]):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05, 
             f'{val:.2f}%', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig_overload_by_season.png", dpi=300, bbox_inches="tight")
plt.show()

print("\nSaved: /dbfs/tmp/thesis_fig_overload_by_season.png")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Print summary for caption
# ─────────────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("SUMMARY FOR FIGURE CAPTION")
print("="*60)
total_overloads = season_stats["overload_count"].sum()
for _, row in season_stats.iterrows():
    pct = 100 * row["overload_count"] / total_overloads
    print(f"{row['season']:8s}: {row['overload_count']:,} events ({pct:.1f}%), rate = {row['overload_rate']:.2f}%")

In [0]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
import pandas as pd

# Load from gold_dataset (unscaled)
df = spark.read.table("hive_metastore.gold.gold_dataset")

# Filter out zeros and compute load ratios
#df = df.filter((F.col("current") > 0) & (F.col("voltage") > 0))

df = df.withColumn(
    "load_ratio_c",
    F.when(F.col("H_LIM_C") > 0, F.col("current") / F.col("H_LIM_C")).otherwise(None)
).withColumn(
    "load_ratio_v",
    F.when(F.col("H_LIM_V") > 0, F.col("voltage") / F.col("H_LIM_V")).otherwise(None)
)

# Flag overload events
df = df.withColumn(
    "is_overload",
    F.when((F.col("load_ratio_c") > 1) | (F.col("load_ratio_v") > 1), 1).otherwise(0)
)

# Extract month and map to season
df = df.withColumn("month", F.month("DATE"))
df = df.withColumn(
    "season",
    F.when(F.col("month").isin(12, 1, 2), "Winter")
     .when(F.col("month").isin(3, 4, 5), "Spring")
     .when(F.col("month").isin(6, 7, 8), "Summer")
     .otherwise("Autumn")
)

# Aggregate by season
season_stats = (
    df.groupBy("season")
    .agg(F.sum("is_overload").alias("overload_count"))
    .toPandas()
)

# Order seasons correctly
season_order = ["Winter", "Spring", "Summer", "Autumn"]
season_stats["season"] = pd.Categorical(season_stats["season"], categories=season_order, ordered=True)
season_stats = season_stats.sort_values("season")

print(season_stats)

# --- Pie Chart ---
fig, ax = plt.subplots(figsize=(7, 7), dpi=150)

colors = ["#4A90D9", "#7CB342", "#FFA726", "#8D6E63"]

wedges, texts, autotexts = ax.pie(
    season_stats["overload_count"],
    labels=season_stats["season"],
    autopct=lambda pct: f'{pct:.1f}%\n({int(pct/100 * season_stats["overload_count"].sum()):,})',
    colors=colors,
    explode=[0.02] * 4,
    startangle=90,
    textprops={'fontsize': 11}
)

ax.set_title("Overload Events by Season", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig_overload_by_season.png", dpi=300, bbox_inches="tight")
plt.show()

# Print summary
print("\n" + "="*60)
total = season_stats["overload_count"].sum()
for _, row in season_stats.iterrows():
    pct = 100 * row["overload_count"] / total
    print(f"{row['season']:8s}: {row['overload_count']:,} events ({pct:.1f}%)")

In [0]:
import matplotlib.pyplot as plt
from pyspark.sql import functions as F
import pandas as pd

# Load gold_features (has labels)
df = spark.table("hive_metastore.gold.gold_features")

# Map month to season
df = df.withColumn(
    "season",
    F.when(F.col("month").isin(12, 1, 2), "Winter")
     .when(F.col("month").isin(3, 4, 5), "Spring")
     .when(F.col("month").isin(6, 7, 8), "Summer")
     .otherwise("Autumn")
)

# Count positive labels by season
season_stats = (
    df.groupBy("season")
    .agg(
        F.sum(F.col("label_4h").cast("int")).alias("pos_4h"),
        F.sum(F.col("label_24h").cast("int")).alias("pos_24h"),
        F.count("*").alias("total")
    )
    .toPandas()
)

# Order seasons correctly
season_order = ["Winter", "Spring", "Summer", "Autumn"]
season_stats["season"] = pd.Categorical(season_stats["season"], categories=season_order, ordered=True)
season_stats = season_stats.sort_values("season")

print(season_stats)

# --- Two pie charts side by side ---
fig, axes = plt.subplots(1, 2, figsize=(12, 6), dpi=150)

colors = ["#4A90D9", "#7CB342", "#FFA726", "#8D6E63"]

# 4-hour label
ax1 = axes[0]
wedges1, texts1, autotexts1 = ax1.pie(
    season_stats["pos_4h"],
    labels=season_stats["season"],
    autopct=lambda pct: f'{pct:.1f}%\n({int(pct/100 * season_stats["pos_4h"].sum()):,})',
    colors=colors,
    explode=[0.02] * 4,
    startangle=90,
    textprops={'fontsize': 11}
)
ax1.set_title("(a) 4-Hour Label Positives", fontsize=12, fontweight="bold")

# 24-hour label
ax2 = axes[1]
wedges2, texts2, autotexts2 = ax2.pie(
    season_stats["pos_24h"],
    labels=season_stats["season"],
    autopct=lambda pct: f'{pct:.1f}%\n({int(pct/100 * season_stats["pos_24h"].sum()):,})',
    colors=colors,
    explode=[0.02] * 4,
    startangle=90,
    textprops={'fontsize': 11}
)
ax2.set_title("(b) 24-Hour Label Positives", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig_label_by_season.png", dpi=300, bbox_inches="tight")
plt.show()

# Print summary
print("\n" + "="*60)
print("4-HOUR LABEL")
total_4h = season_stats["pos_4h"].sum()
for _, row in season_stats.iterrows():
    pct = 100 * row["pos_4h"] / total_4h
    print(f"{row['season']:8s}: {row['pos_4h']:,} positives ({pct:.1f}%)")

print("\n24-HOUR LABEL")
total_24h = season_stats["pos_24h"].sum()
for _, row in season_stats.iterrows():
    pct = 100 * row["pos_24h"] / total_24h
    print(f"{row['season']:8s}: {row['pos_24h']:,} positives ({pct:.1f}%)")

In [0]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# First, get the actual numbers from gold_features
from pyspark.sql import functions as F

df = spark.table("hive_metastore.gold.gold_features")

# Calculate positive rates by split
split_stats = df.groupBy("split").agg(
    F.count("*").alias("n_rows"),
    F.sum(F.col("label_4h").cast("int")).alias("pos_4h"),
    F.sum(F.col("label_24h").cast("int")).alias("pos_24h")
).withColumn("rate_4h", F.col("pos_4h") / F.col("n_rows") * 100
).withColumn("rate_24h", F.col("pos_24h") / F.col("n_rows") * 100
).orderBy("split")

split_stats.show()

# Collect for plotting
stats = {row["split"]: row.asDict() for row in split_stats.collect()}

# --- Figure ---
fig, ax = plt.subplots(figsize=(8, 5))

splits = ["train", "test"]
x = np.arange(len(splits))
width = 0.35

# Rates
rates_4h = [stats["train"]["rate_4h"], stats["test"]["rate_4h"]]
rates_24h = [stats["train"]["rate_24h"], stats["test"]["rate_24h"]]

bars_4h = ax.bar(x - width/2, rates_4h, width, label="4-hour horizon", color="#2c7fb8")
bars_24h = ax.bar(x + width/2, rates_24h, width, label="24-hour horizon", color="#7fcdbb")

# Labels and formatting
ax.set_ylabel("Positive label rate (%)")
ax.set_xticks(x)
ax.set_xticklabels([
    f"Training\n(Jul 2023 – Mar 2024)\nn = {stats['train']['n_rows']:,}",
    f"Test\n(Apr 2024 – Jul 2024)\nn = {stats['test']['n_rows']:,}"
])
ax.legend(loc="upper right")
ax.set_ylim(0, max(rates_24h) * 1.3)

# Add value labels on bars
for bars in [bars_4h, bars_24h]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.2f}%",
                    xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9)

# Add split date annotation
ax.axhline(y=0, color="black", linewidth=0.8)
fig.text(0.5, 0.02, "Split boundary: 1 April 2024", 
         ha="center", fontsize=9, style="italic")

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.savefig("/dbfs/tmp/thesis_fig_train_test_split.png", dpi=150, bbox_inches="tight")
plt.show()

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Spatial Distribution of Overload Events
# MAGIC 
# MAGIC This notebook computes overload rates by concelho and district for the descriptive statistics section.

# COMMAND ----------

from pyspark.sql import functions as F
from pyspark.sql.types import StringType
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Suppress duplicate figure rendering in Databricks
plt.ioff()

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Concelho to District Mapping
# MAGIC 
# MAGIC Mainland Portugal has 18 districts and 278 concelhos. This mapping covers the major concelhos
# MAGIC you're likely to encounter in the E-REDES network data.

# COMMAND ----------

# District mapping for mainland Portugal concelhos
# This is a representative mapping - extend as needed based on your actual data
CONCELHO_TO_DISTRICT = {
    # Aveiro (19 concelhos)
    "Águeda": "Aveiro", "Albergaria-a-Velha": "Aveiro", "Anadia": "Aveiro",
    "Arouca": "Aveiro", "Aveiro": "Aveiro", "Castelo de Paiva": "Aveiro",
    "Espinho": "Aveiro", "Estarreja": "Aveiro", "Ílhavo": "Aveiro",
    "Mealhada": "Aveiro", "Murtosa": "Aveiro", "Oliveira de Azeméis": "Aveiro",
    "Oliveira do Bairro": "Aveiro", "Ovar": "Aveiro", "Santa Maria da Feira": "Aveiro",
    "São João da Madeira": "Aveiro", "Sever do Vouga": "Aveiro",
    "Vagos": "Aveiro", "Vale de Cambra": "Aveiro",
    
    # Beja (14 concelhos)
    "Aljustrel": "Beja", "Almodôvar": "Beja", "Alvito": "Beja", "Barrancos": "Beja",
    "Beja": "Beja", "Castro Verde": "Beja", "Cuba": "Beja", "Ferreira do Alentejo": "Beja",
    "Mértola": "Beja", "Moura": "Beja", "Odemira": "Beja", "Ourique": "Beja",
    "Serpa": "Beja", "Vidigueira": "Beja",
    
    # Braga (14 concelhos)
    "Amares": "Braga", "Barcelos": "Braga", "Braga": "Braga", "Cabeceiras de Basto": "Braga",
    "Celorico de Basto": "Braga", "Esposende": "Braga", "Fafe": "Braga",
    "Guimarães": "Braga", "Póvoa de Lanhoso": "Braga", "Terras de Bouro": "Braga",
    "Vieira do Minho": "Braga", "Vila Nova de Famalicão": "Braga",
    "Vila Verde": "Braga", "Vizela": "Braga",
    
    # Bragança (12 concelhos)
    "Alfândega da Fé": "Bragança", "Bragança": "Bragança", "Carrazeda de Ansiães": "Bragança",
    "Freixo de Espada à Cinta": "Bragança", "Macedo de Cavaleiros": "Bragança",
    "Miranda do Douro": "Bragança", "Mirandela": "Bragança", "Mogadouro": "Bragança",
    "Torre de Moncorvo": "Bragança", "Vila Flor": "Bragança",
    "Vimioso": "Bragança", "Vinhais": "Bragança",
    
    # Castelo Branco (11 concelhos)
    "Belmonte": "Castelo Branco", "Castelo Branco": "Castelo Branco",
    "Covilhã": "Castelo Branco", "Fundão": "Castelo Branco", "Idanha-a-Nova": "Castelo Branco",
    "Oleiros": "Castelo Branco", "Penamacor": "Castelo Branco", "Proença-a-Nova": "Castelo Branco",
    "Sertã": "Castelo Branco", "Vila de Rei": "Castelo Branco", "Vila Velha de Ródão": "Castelo Branco",
    
    # Coimbra (17 concelhos)
    "Arganil": "Coimbra", "Cantanhede": "Coimbra", "Coimbra": "Coimbra",
    "Condeixa-a-Nova": "Coimbra", "Figueira da Foz": "Coimbra", "Góis": "Coimbra",
    "Lousã": "Coimbra", "Mira": "Coimbra", "Miranda do Corvo": "Coimbra",
    "Montemor-o-Velho": "Coimbra", "Oliveira do Hospital": "Coimbra",
    "Pampilhosa da Serra": "Coimbra", "Penacova": "Coimbra", "Penela": "Coimbra",
    "Soure": "Coimbra", "Tábua": "Coimbra", "Vila Nova de Poiares": "Coimbra",
    
    # Évora (14 concelhos)
    "Alandroal": "Évora", "Arraiolos": "Évora", "Borba": "Évora", "Estremoz": "Évora",
    "Évora": "Évora", "Montemor-o-Novo": "Évora", "Mora": "Évora", "Mourão": "Évora",
    "Portel": "Évora", "Redondo": "Évora", "Reguengos de Monsaraz": "Évora",
    "Vendas Novas": "Évora", "Viana do Alentejo": "Évora", "Vila Viçosa": "Évora",
    
    # Faro (16 concelhos)
    "Albufeira": "Faro", "Alcoutim": "Faro", "Aljezur": "Faro", "Castro Marim": "Faro",
    "Faro": "Faro", "Lagoa": "Faro", "Lagos": "Faro", "Loulé": "Faro",
    "Monchique": "Faro", "Olhão": "Faro", "Portimão": "Faro",
    "São Brás de Alportel": "Faro", "Silves": "Faro", "Tavira": "Faro",
    "Vila do Bispo": "Faro", "Vila Real de Santo António": "Faro",
    
    # Guarda (14 concelhos)
    "Aguiar da Beira": "Guarda", "Almeida": "Guarda", "Celorico da Beira": "Guarda",
    "Figueira de Castelo Rodrigo": "Guarda", "Fornos de Algodres": "Guarda",
    "Gouveia": "Guarda", "Guarda": "Guarda", "Manteigas": "Guarda",
    "Mêda": "Guarda", "Pinhel": "Guarda", "Sabugal": "Guarda",
    "Seia": "Guarda", "Trancoso": "Guarda", "Vila Nova de Foz Côa": "Guarda",
    
    # Leiria (16 concelhos)
    "Alcobaça": "Leiria", "Alvaiázere": "Leiria", "Ansião": "Leiria",
    "Batalha": "Leiria", "Bombarral": "Leiria", "Caldas da Rainha": "Leiria",
    "Castanheira de Pêra": "Leiria", "Figueiró dos Vinhos": "Leiria",
    "Leiria": "Leiria", "Marinha Grande": "Leiria", "Nazaré": "Leiria",
    "Óbidos": "Leiria", "Pedrógão Grande": "Leiria", "Peniche": "Leiria",
    "Pombal": "Leiria", "Porto de Mós": "Leiria",
    
    # Lisboa (16 concelhos)
    "Alenquer": "Lisboa", "Amadora": "Lisboa", "Arruda dos Vinhos": "Lisboa",
    "Azambuja": "Lisboa", "Cadaval": "Lisboa", "Cascais": "Lisboa",
    "Lisboa": "Lisboa", "Loures": "Lisboa", "Lourinhã": "Lisboa",
    "Mafra": "Lisboa", "Odivelas": "Lisboa", "Oeiras": "Lisboa",
    "Sintra": "Lisboa", "Sobral de Monte Agraço": "Lisboa",
    "Torres Vedras": "Lisboa", "Vila Franca de Xira": "Lisboa",
    
    # Portalegre (15 concelhos)
    "Alter do Chão": "Portalegre", "Arronches": "Portalegre", "Avis": "Portalegre",
    "Campo Maior": "Portalegre", "Castelo de Vide": "Portalegre", "Crato": "Portalegre",
    "Elvas": "Portalegre", "Fronteira": "Portalegre", "Gavião": "Portalegre",
    "Marvão": "Portalegre", "Monforte": "Portalegre", "Nisa": "Portalegre",
    "Ponte de Sor": "Portalegre", "Portalegre": "Portalegre", "Sousel": "Portalegre",
    
    # Porto (18 concelhos)
    "Amarante": "Porto", "Baião": "Porto", "Felgueiras": "Porto", "Gondomar": "Porto",
    "Lousada": "Porto", "Maia": "Porto", "Marco de Canaveses": "Porto",
    "Matosinhos": "Porto", "Paços de Ferreira": "Porto", "Paredes": "Porto",
    "Penafiel": "Porto", "Porto": "Porto", "Póvoa de Varzim": "Porto",
    "Santo Tirso": "Porto", "Trofa": "Porto", "Valongo": "Porto",
    "Vila do Conde": "Porto", "Vila Nova de Gaia": "Porto",
    
    # Santarém (21 concelhos)
    "Abrantes": "Santarém", "Alcanena": "Santarém", "Almeirim": "Santarém",
    "Alpiarça": "Santarém", "Benavente": "Santarém", "Cartaxo": "Santarém",
    "Chamusca": "Santarém", "Constância": "Santarém", "Coruche": "Santarém",
    "Entroncamento": "Santarém", "Ferreira do Zêzere": "Santarém", "Golegã": "Santarém",
    "Mação": "Santarém", "Ourém": "Santarém", "Rio Maior": "Santarém",
    "Salvaterra de Magos": "Santarém", "Santarém": "Santarém", "Sardoal": "Santarém",
    "Tomar": "Santarém", "Torres Novas": "Santarém", "Vila Nova da Barquinha": "Santarém",
    
    # Setúbal (13 concelhos)
    "Alcácer do Sal": "Setúbal", "Alcochete": "Setúbal", "Almada": "Setúbal",
    "Barreiro": "Setúbal", "Grândola": "Setúbal", "Moita": "Setúbal",
    "Montijo": "Setúbal", "Palmela": "Setúbal", "Santiago do Cacém": "Setúbal",
    "Seixal": "Setúbal", "Sesimbra": "Setúbal", "Setúbal": "Setúbal", "Sines": "Setúbal",
    
    # Viana do Castelo (10 concelhos)
    "Arcos de Valdevez": "Viana do Castelo", "Caminha": "Viana do Castelo",
    "Melgaço": "Viana do Castelo", "Monção": "Viana do Castelo",
    "Paredes de Coura": "Viana do Castelo", "Ponte da Barca": "Viana do Castelo",
    "Ponte de Lima": "Viana do Castelo", "Valença": "Viana do Castelo",
    "Viana do Castelo": "Viana do Castelo", "Vila Nova de Cerveira": "Viana do Castelo",
    
    # Vila Real (14 concelhos)
    "Alijó": "Vila Real", "Boticas": "Vila Real", "Chaves": "Vila Real",
    "Mesão Frio": "Vila Real", "Mondim de Basto": "Vila Real", "Montalegre": "Vila Real",
    "Murça": "Vila Real", "Peso da Régua": "Vila Real", "Ribeira de Pena": "Vila Real",
    "Sabrosa": "Vila Real", "Santa Marta de Penaguião": "Vila Real",
    "Valpaços": "Vila Real", "Vila Pouca de Aguiar": "Vila Real", "Vila Real": "Vila Real",
    
    # Viseu (24 concelhos)
    "Armamar": "Viseu", "Carregal do Sal": "Viseu", "Castro Daire": "Viseu",
    "Cinfães": "Viseu", "Lamego": "Viseu", "Mangualde": "Viseu",
    "Moimenta da Beira": "Viseu", "Mortágua": "Viseu", "Nelas": "Viseu",
    "Oliveira de Frades": "Viseu", "Penalva do Castelo": "Viseu",
    "Penedono": "Viseu", "Resende": "Viseu", "Santa Comba Dão": "Viseu",
    "São João da Pesqueira": "Viseu", "São Pedro do Sul": "Viseu",
    "Sátão": "Viseu", "Sernancelhe": "Viseu", "Tabuaço": "Viseu",
    "Tarouca": "Viseu", "Tondela": "Viseu", "Vila Nova de Paiva": "Viseu",
    "Viseu": "Viseu", "Vouzela": "Viseu",
}

# Create broadcast variable for efficient joins
concelho_district_map = spark.sparkContext.broadcast(CONCELHO_TO_DISTRICT)

# UDF to map concelho to district
@F.udf(StringType())
def get_district(concelho):
    if concelho is None:
        return "Unknown"
    # Try exact match first
    if concelho in concelho_district_map.value:
        return concelho_district_map.value[concelho]
    # Try case-insensitive match
    concelho_lower = concelho.lower()
    for key, district in concelho_district_map.value.items():
        if key.lower() == concelho_lower:
            return district
    return "Unknown"

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Load Gold Dataset and Compute Overload Metrics

# COMMAND ----------

# Load gold dataset table
gold_dataset = spark.read.table("hive_metastore.gold.gold_dataset")

# Check available columns
print("Available columns:")
print(gold_dataset.columns)

# COMMAND ----------

# Compute load ratios and overload flags from raw measurements
df = gold_dataset.select(
    "DATE",
    "ID_prefix", 
    "concelho",
    "current",
    "voltage",
    "H_LIM_C",
    "H_LIM_V"
).withColumn(
    "load_ratio_c",
    F.when(F.col("H_LIM_C") > 0, F.col("current") / F.col("H_LIM_C")).otherwise(None)
).withColumn(
    "load_ratio_v",
    F.when(F.col("H_LIM_V") > 0, F.col("voltage") / F.col("H_LIM_V")).otherwise(None)
).withColumn(
    "overload_c", F.when(F.col("load_ratio_c") > 1, 1).otherwise(0)
).withColumn(
    "overload_v", F.when(F.col("load_ratio_v") > 1, 1).otherwise(0)
).withColumn(
    "overload_any", F.when((F.col("load_ratio_c") > 1) | (F.col("load_ratio_v") > 1), 1).otherwise(0)
).withColumn(
    "district", get_district(F.col("concelho"))
)

# Cache for multiple aggregations
df.cache()

print(f"Total records: {df.count():,}")
print(f"Distinct concelhos: {df.select('concelho').distinct().count()}")
print(f"Distinct districts: {df.select('district').distinct().count()}")

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Aggregate by Concelho

# COMMAND ----------

# Overload rates by concelho
concelho_stats = df.groupBy("concelho", "district").agg(
    F.count("*").alias("total_measurements"),
    F.countDistinct("ID_prefix").alias("transformer_count"),
    F.sum("overload_c").alias("current_overloads"),
    F.sum("overload_v").alias("voltage_overloads"),
    F.sum("overload_any").alias("any_overloads")
).withColumn(
    "current_overload_rate", F.col("current_overloads") / F.col("total_measurements") * 100
).withColumn(
    "voltage_overload_rate", F.col("voltage_overloads") / F.col("total_measurements") * 100
).withColumn(
    "any_overload_rate", F.col("any_overloads") / F.col("total_measurements") * 100
).orderBy(F.desc("any_overloads"))

concelho_stats.cache()
display(concelho_stats)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Aggregate by District

# COMMAND ----------

# District-level summary
district_stats = df.groupBy("district").agg(
    F.count("*").alias("total_measurements"),
    F.countDistinct("ID_prefix").alias("transformer_count"),
    F.countDistinct("concelho").alias("concelho_count"),
    F.sum("overload_c").alias("current_overloads"),
    F.sum("overload_v").alias("voltage_overloads"),
    F.sum("overload_any").alias("any_overloads")
).withColumn(
    "current_overload_rate", F.col("current_overloads") / F.col("total_measurements") * 100
).withColumn(
    "voltage_overload_rate", F.col("voltage_overloads") / F.col("total_measurements") * 100
).withColumn(
    "any_overload_rate", F.col("any_overloads") / F.col("total_measurements") * 100
).orderBy(F.desc("any_overloads"))

display(district_stats)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Visualisation: Top Concelhos by Overload Count

# COMMAND ----------

# Convert to pandas for plotting
concelho_pdf = concelho_stats.filter(F.col("concelho").isNotNull()).toPandas()

# Sort and take top 20 concelhos by total overload events
top_concelhos = concelho_pdf.nlargest(20, 'any_overloads').sort_values('any_overloads', ascending=True)

# Define district color palette (colorblind-friendly)
districts = top_concelhos['district'].unique()
# Okabe-Ito palette extended
colors = ['#E69F00', '#56B4E9', '#009E73', '#F0E442', '#0072B2', 
          '#D55E00', '#CC79A7', '#000000', '#999999', '#661100',
          '#6699CC', '#994455', '#22AAAA', '#EECC66', '#88CCEE',
          '#44AA99', '#117733', '#882255']
district_colors = {d: colors[i % len(colors)] for i, d in enumerate(sorted(districts))}

# Create figure
plt.ioff()
fig, ax = plt.subplots(figsize=(10, 8))

# Horizontal bar chart
bars = ax.barh(
    top_concelhos['concelho'],
    top_concelhos['any_overloads'],
    color=[district_colors[d] for d in top_concelhos['district']],
    edgecolor='white',
    linewidth=0.5
)

# Formatting
ax.set_xlabel('Total Overload Events', fontsize=10)
ax.set_ylabel('')
ax.set_title('Top 20 Concelhos by Overload Event Count', fontsize=11, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add value labels
for bar, val in zip(bars, top_concelhos['any_overloads']):
    ax.text(bar.get_width() + max(top_concelhos['any_overloads']) * 0.01, 
            bar.get_y() + bar.get_height()/2,
            f'{int(val):,}', va='center', fontsize=8)

# Create legend for districts
legend_patches = [mpatches.Patch(color=district_colors[d], label=d) 
                  for d in sorted(districts)]
ax.legend(handles=legend_patches, title='District', loc='lower right', 
          fontsize=8, title_fontsize=9, frameon=False)

plt.tight_layout()
plt.ion()
display(fig)

# Save figure
fig.savefig('/tmp/concelho_overloads_barplot.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig('/tmp/concelho_overloads_barplot.pdf', bbox_inches='tight', facecolor='white')

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Visualisation: District-Level Summary

# COMMAND ----------

district_pdf = district_stats.filter(F.col("district") != "Unknown").toPandas()
district_pdf = district_pdf.sort_values('any_overloads', ascending=True)

plt.ioff()
fig, axes = plt.subplots(1, 2, figsize=(12, 6))

# Panel (a): Total overload events by district
ax1 = axes[0]
colors_bar = ['#0072B2' if d not in ['Lisboa', 'Porto', 'Setúbal'] else '#D55E00' 
              for d in district_pdf['district']]
bars1 = ax1.barh(district_pdf['district'], district_pdf['any_overloads'], color=colors_bar)
ax1.set_xlabel('Total Overload Events', fontsize=10)
ax1.set_title('(a) Total overload events by district', fontsize=11)
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Add value labels
for bar, val in zip(bars1, district_pdf['any_overloads']):
    ax1.text(bar.get_width() + max(district_pdf['any_overloads']) * 0.01,
             bar.get_y() + bar.get_height()/2,
             f'{int(val):,}', va='center', fontsize=7)

# Panel (b): Overload rate by district
ax2 = axes[1]
bars2 = ax2.barh(district_pdf['district'], district_pdf['any_overload_rate'], color='#56B4E9')
ax2.set_xlabel('Overload Rate (%)', fontsize=10)
ax2.set_title('(b) Overload rate by district', fontsize=11)
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Add value labels
for bar, val in zip(bars2, district_pdf['any_overload_rate']):
    ax2.text(bar.get_width() + max(district_pdf['any_overload_rate']) * 0.02,
             bar.get_y() + bar.get_height()/2,
             f'{val:.2f}%', va='center', fontsize=7)

plt.tight_layout()
plt.ion()
display(fig)

fig.savefig('/tmp/district_overloads_summary.png', dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig('/tmp/district_overloads_summary.pdf', bbox_inches='tight', facecolor='white')

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Summary Statistics for Thesis Text

# COMMAND ----------

# Key statistics for thesis narrative
total_overloads_c = df.agg(F.sum("overload_c")).collect()[0][0]
total_overloads_v = df.agg(F.sum("overload_v")).collect()[0][0]
total_measurements = df.count()

top_3_concelhos = concelho_pdf.nlargest(3, 'any_overloads')[['concelho', 'district', 'any_overloads', 'any_overload_rate']]
top_3_districts = district_pdf.nlargest(3, 'any_overloads')[['district', 'any_overloads', 'transformer_count']]

# Concentration metrics
total_overloads = concelho_pdf['any_overloads'].sum()
top_10_share = concelho_pdf.nlargest(10, 'any_overloads')['any_overloads'].sum() / total_overloads * 100
top_3_share = concelho_pdf.nlargest(3, 'any_overloads')['any_overloads'].sum() / total_overloads * 100

print("="*60)
print("SUMMARY STATISTICS FOR THESIS")
print("="*60)
print(f"\nTotal measurements: {total_measurements:,}")
print(f"Total current overloads: {total_overloads_c:,}")
print(f"Total voltage overloads: {total_overloads_v:,}")
print(f"\nTop 3 concelhos account for {top_3_share:.1f}% of all overloads")
print(f"Top 10 concelhos account for {top_10_share:.1f}% of all overloads")
print(f"\nTop 3 concelhos by overload count:")
print(top_3_concelhos.to_string(index=False))
print(f"\nTop 3 districts by overload count:")
print(top_3_districts.to_string(index=False))
print(f"\nNumber of concelhos with data: {len(concelho_pdf)}")
print(f"Number of districts with data: {len(district_pdf)}")

# COMMAND ----------

# Clean up
df.unpersist()
concelho_stats.unpersist()

In [0]:
from pyspark.sql import functions as F

spark.read.table("hive_metastore.gold.gold_dataset") \
    .filter(F.col("concelho").isin("Lisboa", "Porto")) \
    .withColumn("overload_c", F.when((F.col("H_LIM_C") > 0) & (F.col("current") / F.col("H_LIM_C") > 1), 1).otherwise(0)) \
    .withColumn("overload_v", F.when((F.col("H_LIM_V") > 0) & (F.col("voltage") / F.col("H_LIM_V") > 1), 1).otherwise(0)) \
    .groupBy("concelho") \
    .agg(
        F.count("*").alias("total_measurements"),
        F.sum("overload_c").alias("current_overloads"),
        F.sum("overload_v").alias("voltage_overloads")
    ) \
    .show()

In [0]:
%sql
SELECT 
    CASE 
        WHEN concelho IN ('Alenquer', 'Amadora', 'Arruda dos Vinhos', 'Azambuja', 'Cadaval', 
                          'Cascais', 'Lisboa', 'Loures', 'Lourinhã', 'Mafra', 'Odivelas', 
                          'Oeiras', 'Sintra', 'Sobral de Monte Agraço', 'Torres Vedras', 
                          'Vila Franca de Xira') THEN 'Lisboa'
        WHEN concelho IN ('Amarante', 'Baião', 'Felgueiras', 'Gondomar', 'Lousada', 'Maia',
                          'Marco de Canaveses', 'Matosinhos', 'Paços de Ferreira', 'Paredes',
                          'Penafiel', 'Porto', 'Póvoa de Varzim', 'Santo Tirso', 'Trofa',
                          'Valongo', 'Vila do Conde', 'Vila Nova de Gaia') THEN 'Porto'
    END AS district,
    concelho,
    COUNT(*) AS total_measurements,
    SUM(CASE WHEN H_LIM_C > 0 AND current / H_LIM_C > 1 THEN 1 ELSE 0 END) AS current_overloads,
    SUM(CASE WHEN H_LIM_V > 0 AND voltage / H_LIM_V > 1 THEN 1 ELSE 0 END) AS voltage_overloads
FROM gold.gold_dataset
WHERE concelho IN (
    -- Lisboa district
    'Alenquer', 'Amadora', 'Arruda dos Vinhos', 'Azambuja', 'Cadaval', 'Cascais', 
    'Lisboa', 'Loures', 'Lourinhã', 'Mafra', 'Odivelas', 'Oeiras', 'Sintra', 
    'Sobral de Monte Agraço', 'Torres Vedras', 'Vila Franca de Xira',
    -- Porto district
    'Amarante', 'Baião', 'Felgueiras', 'Gondomar', 'Lousada', 'Maia',
    'Marco de Canaveses', 'Matosinhos', 'Paços de Ferreira', 'Paredes',
    'Penafiel', 'Porto', 'Póvoa de Varzim', 'Santo Tirso', 'Trofa',
    'Valongo', 'Vila do Conde', 'Vila Nova de Gaia'
)
GROUP BY concelho
ORDER BY district, voltage_overloads DESC

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("gold.gold_dataset")

# How many transformers are actually monitored per district?
df.withColumn("district", get_district(F.col("concelho"))) \
    .groupBy("district") \
    .agg(
        F.countDistinct("ID_prefix").alias("transformer_count"),
        F.count("*").alias("measurements"),
        F.sum(F.when(F.col("H_LIM_C").isNull() | (F.col("H_LIM_C") == 0), 1).otherwise(0)).alias("missing_hlim_c"),
        F.sum(F.when(F.col("H_LIM_V").isNull() | (F.col("H_LIM_V") == 0), 1).otherwise(0)).alias("missing_hlim_v")
    ) \
    .orderBy(F.desc("transformer_count")) \
    .show(25, truncate=False)

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

CONCELHO_TO_DISTRICT = {
    # Aveiro
    'Águeda': 'Aveiro', 'Albergaria-a-Velha': 'Aveiro', 'Anadia': 'Aveiro', 'Arouca': 'Aveiro',
    'Aveiro': 'Aveiro', 'Castelo de Paiva': 'Aveiro', 'Espinho': 'Aveiro', 'Estarreja': 'Aveiro',
    'Ílhavo': 'Aveiro', 'Mealhada': 'Aveiro', 'Oliveira de Azeméis': 'Aveiro',
    'Oliveira do Bairro': 'Aveiro', 'Ovar': 'Aveiro', 'Santa Maria da Feira': 'Aveiro',
    'São João da Madeira': 'Aveiro', 'Vale de Cambra': 'Aveiro',
    # Beja
    'Aljustrel': 'Beja', 'Almodôvar': 'Beja', 'Beja': 'Beja', 'Castro Verde': 'Beja',
    'Ferreira do Alentejo': 'Beja', 'Mértola': 'Beja', 'Moura': 'Beja', 'Odemira': 'Beja',
    'Ourique': 'Beja', 'Serpa': 'Beja', 'Vidigueira': 'Beja',
    # Braga
    'Amares': 'Braga', 'Barcelos': 'Braga', 'Braga': 'Braga', 'Celorico de Basto': 'Braga',
    'Esposende': 'Braga', 'Fafe': 'Braga', 'Guimarães': 'Braga', 'Vieira do Minho': 'Braga',
    'Vila Nova de Famalicão': 'Braga', 'Vila Verde': 'Braga',
    # Bragança
    'Bragança': 'Bragança', 'Macedo de Cavaleiros': 'Bragança', 'Mirandela': 'Bragança',
    'Mogadouro': 'Bragança',
    # Castelo Branco
    'Belmonte': 'Castelo Branco', 'Castelo Branco': 'Castelo Branco', 'Covilhã': 'Castelo Branco',
    'Idanha-a-Nova': 'Castelo Branco', 'Oleiros': 'Castelo Branco', 'Penamacor': 'Castelo Branco',
    'Proença-a-Nova': 'Castelo Branco', 'Sertã': 'Castelo Branco', 'Vila Velha de Rodão': 'Castelo Branco',
    # Coimbra
    'Arganil': 'Coimbra', 'Cantanhede': 'Coimbra', 'Coimbra': 'Coimbra', 'Figueira da Foz': 'Coimbra',
    'Góis': 'Coimbra', 'Lousã': 'Coimbra', 'Mira': 'Coimbra', 'Miranda do Corvo': 'Coimbra',
    'Oliveira do Hospital': 'Coimbra', 'Pampilhosa da Serra': 'Coimbra', 'Penacova': 'Coimbra',
    'Soure': 'Coimbra', 'Tábua': 'Coimbra',
    # Évora
    'Arraiolos': 'Évora', 'Borba': 'Évora', 'Estremoz': 'Évora', 'Évora': 'Évora',
    'Montemor-o-Novo': 'Évora', 'Portel': 'Évora', 'Reguengos de Monsaraz': 'Évora',
    'Vendas Novas': 'Évora', 'Viana do Alentejo': 'Évora', 'Vila Viçosa': 'Évora',
    # Faro
    'Albufeira': 'Faro', 'Aljezur': 'Faro', 'Castro Marim': 'Faro', 'Faro': 'Faro',
    'Lagoa': 'Faro', 'Lagos': 'Faro', 'Loulé': 'Faro', 'Monchique': 'Faro',
    'Olhão': 'Faro', 'Portimão': 'Faro', 'São Brás de Alportel': 'Faro', 'Silves': 'Faro',
    'Tavira': 'Faro',
    # Guarda
    'Celorico da Beira': 'Guarda', 'Guarda': 'Guarda', 'Pinhel': 'Guarda',
    'Sabugal': 'Guarda', 'Seia': 'Guarda', 'Trancoso': 'Guarda',
    # Leiria
    'Alcobaça': 'Leiria', 'Alvaiázere': 'Leiria', 'Ansião': 'Leiria', 'Batalha': 'Leiria',
    'Caldas da Rainha': 'Leiria', 'Leiria': 'Leiria', 'Marinha Grande': 'Leiria',
    'Óbidos': 'Leiria', 'Pedrógão Grande': 'Leiria', 'Peniche': 'Leiria', 'Pombal': 'Leiria',
    'Porto de Mós': 'Leiria',
    # Lisboa
    'Alenquer': 'Lisboa', 'Amadora': 'Lisboa', 'Azambuja': 'Lisboa', 'Cadaval': 'Lisboa',
    'Cascais': 'Lisboa', 'Lisboa': 'Lisboa', 'Loures': 'Lisboa', 'Lourinhã': 'Lisboa',
    'Mafra': 'Lisboa', 'Odivelas': 'Lisboa', 'Oeiras': 'Lisboa', 'Sintra': 'Lisboa',
    'Torres Vedras': 'Lisboa', 'Vila Franca de Xira': 'Lisboa',
    # Portalegre
    'Alter do Chão': 'Portalegre', 'Arronches': 'Portalegre', 'Avis': 'Portalegre',
    'Elvas': 'Portalegre', 'Nisa': 'Portalegre', 'Ponte de Sôr': 'Portalegre',
    'Portalegre': 'Portalegre',
    # Porto
    'Amarante': 'Porto', 'Baião': 'Porto', 'Felgueiras': 'Porto', 'Gondomar': 'Porto',
    'Lousada': 'Porto', 'Maia': 'Porto', 'Marco de Canaveses': 'Porto', 'Matosinhos': 'Porto',
    'Paços de Ferreira': 'Porto', 'Paredes': 'Porto', 'Penafiel': 'Porto', 'Porto': 'Porto',
    'Póvoa de Varzim': 'Porto', 'Santo Tirso': 'Porto', 'Trofa': 'Porto', 'Valongo': 'Porto',
    'Vila do Conde': 'Porto', 'Vila Nova de Gaia': 'Porto',
    # Santarém
    'Abrantes': 'Santarém', 'Alcanena': 'Santarém', 'Almeirim': 'Santarém', 'Benavente': 'Santarém',
    'Cartaxo': 'Santarém', 'Coruche': 'Santarém', 'Entroncamento': 'Santarém', 'Mação': 'Santarém',
    'Ourém': 'Santarém', 'Rio Maior': 'Santarém', 'Salvaterra de Magos': 'Santarém',
    'Santarém': 'Santarém', 'Tomar': 'Santarém', 'Torres Novas': 'Santarém',
    'Vila Nova da Barquinha': 'Santarém',
    # Setúbal
    'Alcácer do Sal': 'Setúbal', 'Alcochete': 'Setúbal', 'Almada': 'Setúbal', 'Barreiro': 'Setúbal',
    'Moita': 'Setúbal', 'Montijo': 'Setúbal', 'Palmela': 'Setúbal', 'Santiago doCacém': 'Setúbal',
    'Seixal': 'Setúbal', 'Sesimbra': 'Setúbal', 'Setúbal': 'Setúbal', 'Sines': 'Setúbal',
    # Viana do Castelo
    'Arcos de Valdevez': 'Viana do Castelo', 'Caminha': 'Viana do Castelo', 'Monção': 'Viana do Castelo',
    'Ponte da Barca': 'Viana do Castelo', 'Ponte de Lima': 'Viana do Castelo',
    'Valença': 'Viana do Castelo', 'Viana do Castelo': 'Viana do Castelo',
    'Vila Nova de Cerveira': 'Viana do Castelo',
    # Vila Real
    'Alijó': 'Vila Real', 'Boticas': 'Vila Real', 'Chaves': 'Vila Real', 'Mondim de Basto': 'Vila Real',
    'Montalegre': 'Vila Real', 'Ribeira de Pena': 'Vila Real', 'Valpaços': 'Vila Real',
    'Vila Pouca de Aguiar': 'Vila Real', 'Vila Real': 'Vila Real',
    # Viseu
    'Carregal do Sal': 'Viseu', 'Castro Daire': 'Viseu', 'Cinfães': 'Viseu', 'Lamego': 'Viseu',
    'Mangualde': 'Viseu', 'Moimenta da Beira': 'Viseu', 'Mortágua': 'Viseu', 'Nelas': 'Viseu',
    'Sátão': 'Viseu', 'Tondela': 'Viseu', 'Viseu': 'Viseu', 'Vouzela': 'Viseu',
    # Unknown
    'NaN': 'Unknown'
}

df = spark.table("hive_metastore.gold.gold_dataset")
df = df.filter((F.col("current") > 0) & (F.col("voltage") > 0))

map_udf = F.udf(lambda x: CONCELHO_TO_DISTRICT.get(x, "Unknown"), StringType())
df = df.withColumn("district", map_udf(F.col("CONCELHO")))

# Build overload flag with explicit guards
cond_c = (F.col("H_LIM_C") > 0) & (F.col("current") > F.col("H_LIM_C"))
cond_v = (F.col("H_LIM_V") > 0) & (F.col("voltage") > F.col("H_LIM_V"))
df = df.withColumn("is_overload", F.when(cond_c | cond_v, 1).otherwise(0))

# Aggregate
agg = df.groupBy("district").agg(
    F.countDistinct("ID_prefix").alias("n_transformers"),
    F.count("*").alias("n_rows"),
    F.sum("is_overload").alias("n_overloads")
)

rate_expr = F.when(F.col("n_rows") > 0, F.round(F.col("n_overloads") / F.col("n_rows") * 100, 3)).otherwise(F.lit(0.0))
agg = agg.withColumn("overload_rate", rate_expr).orderBy(F.desc("n_overloads"))

district_stats = agg.toPandas()

print("="*80)
print("OVERLOADS BY DISTRICT")
print("="*80)
print(district_stats.to_string(index=False))

print("\n" + "="*80)
print("SUMMARY")
print("="*80)
total_overloads = district_stats["n_overloads"].sum()
total_transformers = district_stats["n_transformers"].sum()
print(f"Total districts: {len(district_stats)}")
print(f"Total transformers: {total_transformers:,}")
print(f"Total overloads: {total_overloads:,}")

top3 = district_stats.head(3)
top3_overloads = top3["n_overloads"].sum()
print(f"\nTop 3 districts: {', '.join(top3['district'].tolist())}")
print(f"Top 3 account for: {top3_overloads:,} ({100*top3_overloads/total_overloads:.1f}%)")

In [0]:
from pyspark.sql import functions as F

# First, let's see what CONCELHOs we have
df = spark.table("hive_metastore.gold.gold_dataset")

concelhos = df.select("CONCELHO").distinct().orderBy("CONCELHO").toPandas()
print(f"Unique CONCELHOs: {len(concelhos)}")
print(concelhos["CONCELHO"].tolist())

In [0]:
# =============================================================================
# Thesis §3.4 — All Descriptive Statistics Figures
# =============================================================================
# Source: hive_metastore.gold.gold_dataset (unscaled values)
# All figures filter out rows where current=0 or voltage=0
# Run each CELL block separately in Databricks
# =============================================================================

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pandas as pd
from pyspark.sql import functions as F

SRC_TABLE = "hive_metastore.gold.gold_dataset"
TS_COL = "DATE"

# ─── SHARED DATA LOAD ────────────────────────────────────────────────────────
# Run this cell first — all figures depend on it

df = spark.read.table(SRC_TABLE)
df = df.filter((F.col("current") > 0) & (F.col("voltage") > 0))

df = df.withColumn(
    "load_ratio_c",
    F.when(F.col("H_LIM_C") > 0, F.col("current") / F.col("H_LIM_C")).otherwise(None)
).withColumn(
    "load_ratio_v",
    F.when(F.col("H_LIM_V") > 0, F.col("voltage") / F.col("H_LIM_V")).otherwise(None)
)

print(f"Rows after zero filter: {df.count():,}")


# =============================================================================
# CELL 1 — Figure 4: Fleet-Wide Daily Average Load Ratios (2-panel)
# =============================================================================

daily_avg = (
    df.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(
        F.mean("load_ratio_c").alias("mean_lr_c"),
        F.mean("load_ratio_v").alias("mean_lr_v"),
    )
    .orderBy("date")
    .toPandas()
)
daily_avg["date"] = pd.to_datetime(daily_avg["date"])
daily_avg = daily_avg.set_index("date").sort_index()

fig, axes = plt.subplots(2, 1, figsize=(12, 7), dpi=150, sharex=True)

# Panel A — Current
ax1 = axes[0]
ax1.plot(daily_avg.index, daily_avg["mean_lr_c"], linewidth=0.8, color="#1f77b4", alpha=0.7, label="Daily mean")
ax1.plot(daily_avg.index, daily_avg["mean_lr_c"].rolling(7, center=True).mean(), linewidth=2, color="#1f77b4", label="7-day rolling mean")
ax1.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")
ax1.set_ylabel("Load Ratio (I / I_limit)", fontsize=11)
ax1.set_ylim(0, min(daily_avg["mean_lr_c"].max() * 1.3, 1.2))
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("(a) Current Load Ratio — Daily Average Across All Transformers", fontsize=11, fontweight="bold")

# Panel B — Voltage
ax2 = axes[1]
ax2.plot(daily_avg.index, daily_avg["mean_lr_v"], linewidth=0.8, color="#ff7f0e", alpha=0.7, label="Daily mean")
ax2.plot(daily_avg.index, daily_avg["mean_lr_v"].rolling(7, center=True).mean(), linewidth=2, color="#ff7f0e", label="7-day rolling mean")
ax2.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("Load Ratio (V / V_limit)", fontsize=11)
ax2.set_ylim(0, min(daily_avg["mean_lr_v"].max() * 1.3, 1.2))
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title("(b) Voltage Load Ratio — Daily Average Across All Transformers", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig4_avg_load_ratios.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: /dbfs/tmp/thesis_fig4_avg_load_ratios.png")


# =============================================================================
# CELL 2 — Figure 5: Single Transformer Full Year (BSCVER3TP1-0)
# =============================================================================

TRANSFORMER_ID = "BSCVER3TP1-0"

df_single = (
    df.filter(F.col("ID_prefix") == TRANSFORMER_ID)
    .select(TS_COL, "load_ratio_c")
    .orderBy(TS_COL)
    .toPandas()
)
df_single[TS_COL] = pd.to_datetime(df_single[TS_COL])
df_single = df_single.set_index(TS_COL).sort_index()

fig, ax = plt.subplots(figsize=(12, 4), dpi=150)
ax.plot(df_single.index, df_single["load_ratio_c"], linewidth=0.3, color="#1f77b4", alpha=0.7, label="Load ratio (current)")
ax.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

overload_mask = df_single["load_ratio_c"] > 1
ax.scatter(df_single.index[overload_mask], df_single.loc[overload_mask, "load_ratio_c"],
           color="#d62728", s=3, alpha=0.6, zorder=5, label="Overload events")

ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Load Ratio (I / I_limit)", fontsize=11)
ax.set_ylim(0, min(df_single["load_ratio_c"].max() * 1.1, 3.0))
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_title(f"Transformer {TRANSFORMER_ID} — Load Ratio Over 12 Months", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig5_full_year.png", dpi=300, bbox_inches="tight")
plt.show()

n_overloads = overload_mask.sum()
n_total = len(df_single)
print(f"Saved: /dbfs/tmp/thesis_fig5_full_year.png")
print(f"Overloads: {n_overloads:,} / {n_total:,} ({100*n_overloads/n_total:.2f}%)")


# =============================================================================
# CELL 3 — Figure 6: Single Transformer 2-Week Zoom (1–15 Jan 2024)
# =============================================================================

zoom_start = "2024-01-01"
zoom_end = "2024-01-15"
df_zoom = df_single.loc[zoom_start:zoom_end]

fig, ax = plt.subplots(figsize=(12, 4), dpi=150)
ax.plot(df_zoom.index, df_zoom["load_ratio_c"], linewidth=0.8, color="#1f77b4", alpha=0.9, label="Load ratio (current)")
ax.axhline(y=1.0, color="#d62728", linestyle="--", linewidth=1.5, label="Overload threshold")

overload_mask_zoom = df_zoom["load_ratio_c"] > 1
if overload_mask_zoom.sum() > 0:
    ax.scatter(df_zoom.index[overload_mask_zoom], df_zoom.loc[overload_mask_zoom, "load_ratio_c"],
               color="#d62728", s=20, alpha=0.8, zorder=5, label="Overload events")
    for idx in df_zoom.index[overload_mask_zoom]:
        ax.axvspan(idx, idx + pd.Timedelta(minutes=15), color="#d62728", alpha=0.1)

ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Load Ratio (I / I_limit)", fontsize=11)
ax.set_ylim(0, min(df_zoom["load_ratio_c"].max() * 1.1, 2.5))
ax.xaxis.set_major_locator(mdates.DayLocator(interval=2))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b"))
ax.xaxis.set_minor_locator(mdates.HourLocator(interval=12))
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)
ax.set_title(f"Transformer {TRANSFORMER_ID} — 2-Week Detail (1–15 January 2024)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig6_zoom.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: /dbfs/tmp/thesis_fig6_zoom.png  |  Overloads in window: {overload_mask_zoom.sum()}")


# =============================================================================
# CELL 4 — Figure 7: Weather 4-Panel
# =============================================================================

WEATHER_COLS = [
    "temperatura_media_do_ar_horaria_c",
    "humidade_relativa_media_horaria_percent",
    "precipitacao_horaria_mm",
    "velocidade_do_vento_media_horaria_m_per_s"
]

daily_weather = (
    df.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(*[F.mean(c).alias(c) for c in WEATHER_COLS])
    .orderBy("date")
    .toPandas()
)
daily_weather["date"] = pd.to_datetime(daily_weather["date"])
daily_weather = daily_weather.set_index("date").sort_index()

labels = ["(a) Mean Air Temperature (°C)", "(b) Relative Humidity (%)", "(c) Precipitation (mm)", "(d) Wind Speed (m/s)"]
colors = ["#d62728", "#1f77b4", "#2ca02c", "#9467bd"]

fig, axes = plt.subplots(4, 1, figsize=(12, 10), dpi=150, sharex=True)

for i, (col, label, color) in enumerate(zip(WEATHER_COLS, labels, colors)):
    ax = axes[i]
    ax.plot(daily_weather.index, daily_weather[col], linewidth=0.8, color=color, alpha=0.7)
    rolling = daily_weather[col].rolling(7, center=True).mean()
    ax.plot(daily_weather.index, rolling, linewidth=2, color=color)
    ax.set_ylabel(label.split(") ")[1], fontsize=10)
    ax.set_title(label, fontsize=11, fontweight="bold")
    ax.grid(True, alpha=0.3)

axes[-1].set_xlabel("Date", fontsize=11)
axes[-1].xaxis.set_major_locator(mdates.MonthLocator())
axes[-1].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig7_weather.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: /dbfs/tmp/thesis_fig7_weather.png")


# =============================================================================
# CELL 5 — Figure 8: Daily Overload Rate (Current vs Voltage) + Train/Test Split
# =============================================================================

daily_overload = (
    df.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(
        F.mean(F.when(F.col("load_ratio_c") > 1, 1).otherwise(0)).alias("rate_c"),
        F.mean(F.when(F.col("load_ratio_v") > 1, 1).otherwise(0)).alias("rate_v"),
    )
    .orderBy("date")
    .toPandas()
)
daily_overload["date"] = pd.to_datetime(daily_overload["date"])
daily_overload = daily_overload.set_index("date").sort_index()
daily_overload["rate_c"] *= 100
daily_overload["rate_v"] *= 100

fig, ax = plt.subplots(figsize=(12, 5), dpi=150)

ax.plot(daily_overload.index, daily_overload["rate_c"].rolling(7, center=True).mean(),
        linewidth=2, color="#1f77b4", label="Current overloads")
ax.plot(daily_overload.index, daily_overload["rate_v"].rolling(7, center=True).mean(),
        linewidth=2, color="#ff7f0e", label="Voltage overloads")

# Train/test split line
split_date = pd.Timestamp("2024-04-01")
ax.axvline(x=split_date, color="black", linestyle="--", linewidth=1.5, label="Train/test split")
ax.text(split_date - pd.Timedelta(days=15), ax.get_ylim()[1] * 0.92, "Training", ha="right", fontsize=10, fontstyle="italic")
ax.text(split_date + pd.Timedelta(days=15), ax.get_ylim()[1] * 0.92, "Test", ha="left", fontsize=10, fontstyle="italic")

ax.set_xlabel("Date", fontsize=11)
ax.set_ylabel("Daily Overload Rate (%)", fontsize=11)
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax.legend(loc="upper right", fontsize=10)
ax.grid(True, alpha=0.3)
ax.set_title("Daily Overload Rate Across All Transformers (7-day rolling average)", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig8_overload_rate.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: /dbfs/tmp/thesis_fig8_overload_rate.png")


# =============================================================================
# CELL 6 — Figure 9: Seasonal Pie Chart (Overload Events)
# =============================================================================

df_season = df.withColumn("month", F.month(TS_COL)).withColumn(
    "season",
    F.when(F.col("month").isin(12, 1, 2), "Winter")
     .when(F.col("month").isin(3, 4, 5), "Spring")
     .when(F.col("month").isin(6, 7, 8), "Summer")
     .otherwise("Autumn")
)

season_stats = (
    df_season.withColumn(
        "is_overload",
        F.when((F.col("load_ratio_c") > 1) | (F.col("load_ratio_v") > 1), 1).otherwise(0)
    )
    .groupBy("season")
    .agg(F.sum("is_overload").alias("overload_count"))
    .toPandas()
)

season_order = ["Winter", "Spring", "Summer", "Autumn"]
season_stats["season"] = pd.Categorical(season_stats["season"], categories=season_order, ordered=True)
season_stats = season_stats.sort_values("season")

fig, ax = plt.subplots(figsize=(7, 7), dpi=150)
colors = ["#4A90D9", "#7CB342", "#FFA726", "#8D6E63"]

wedges, texts, autotexts = ax.pie(
    season_stats["overload_count"],
    labels=season_stats["season"],
    autopct=lambda pct: f'{pct:.1f}%\n({int(pct/100 * season_stats["overload_count"].sum()):,})',
    colors=colors,
    explode=[0.02] * 4,
    startangle=90,
    textprops={'fontsize': 11}
)
ax.set_title("Overload Events by Season", fontsize=12, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig9_seasonal_pie.png", dpi=300, bbox_inches="tight")
plt.show()

total = season_stats["overload_count"].sum()
for _, row in season_stats.iterrows():
    pct = 100 * row["overload_count"] / total
    print(f"{row['season']:8s}: {row['overload_count']:,} events ({pct:.1f}%)")
print(f"Saved: /dbfs/tmp/thesis_fig9_seasonal_pie.png")


# =============================================================================
# CELL 7 — Figure 10: SCADA Event Counts (2-panel)
# =============================================================================
# NOTE: This figure reads from gold_dataset. If events_15m_cnt is not in
# gold_dataset, you may need to read from gold_features (but use raw values,
# not scaled). Adjust SRC as needed.

df_events = spark.read.table(SRC_TABLE)

daily_events = (
    df_events.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(
        F.sum("events_15m_cnt").alias("total_events"),
        F.mean(F.when(F.col("events_15m_cnt") > 0, 1).otherwise(0)).alias("pct_with_events"),
        F.count("*").alias("n_rows"),
    )
    .orderBy("date")
    .toPandas()
)
daily_events["date"] = pd.to_datetime(daily_events["date"])
daily_events = daily_events.set_index("date").sort_index()
daily_events["pct_with_events"] *= 100

fig, axes = plt.subplots(2, 1, figsize=(12, 7), dpi=150, sharex=True)

# Panel A — Total event count
ax1 = axes[0]
ax1.bar(daily_events.index, daily_events["total_events"], width=1, color="#2ca02c", alpha=0.7)
ax1.plot(daily_events.index, daily_events["total_events"].rolling(7, center=True).mean(),
         linewidth=2, color="#006400", label="7-day rolling mean")
ax1.set_ylabel("Total Events", fontsize=11)
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("(a) Daily SCADA Event Count Across All Transformers", fontsize=11, fontweight="bold")

# Panel B — Percentage with events
ax2 = axes[1]
ax2.bar(daily_events.index, daily_events["pct_with_events"], width=1, color="#9467bd", alpha=0.7)
ax2.plot(daily_events.index, daily_events["pct_with_events"].rolling(7, center=True).mean(),
         linewidth=2, color="#5B2C6F", label="7-day rolling mean")
ax2.set_xlabel("Date", fontsize=11)
ax2.set_ylabel("Measurements with Events (%)", fontsize=11)
ax2.xaxis.set_major_locator(mdates.MonthLocator())
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_title("(b) Percentage of Measurements with Associated Events", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig10_events.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: /dbfs/tmp/thesis_fig10_events.png")


# =============================================================================
# BONUS — Figure for §3.3.5: Train/Test Split Label Balance
# =============================================================================
# This uses gold_features (has labels)

import numpy as np

gf = spark.table("hive_metastore.gold.gold_features")
split_stats = gf.groupBy("split").agg(
    F.count("*").alias("n_rows"),
    F.sum(F.col("label_4h").cast("int")).alias("pos_4h"),
    F.sum(F.col("label_24h").cast("int")).alias("pos_24h")
).withColumn("rate_4h", F.col("pos_4h") / F.col("n_rows") * 100
).withColumn("rate_24h", F.col("pos_24h") / F.col("n_rows") * 100
).orderBy("split")

stats = {row["split"]: row.asDict() for row in split_stats.collect()}

fig, ax = plt.subplots(figsize=(8, 5))
splits = ["train", "test"]
x = np.arange(len(splits))
width = 0.35

rates_4h = [stats["train"]["rate_4h"], stats["test"]["rate_4h"]]
rates_24h = [stats["train"]["rate_24h"], stats["test"]["rate_24h"]]

bars_4h = ax.bar(x - width/2, rates_4h, width, label="4-hour horizon", color="#2c7fb8")
bars_24h = ax.bar(x + width/2, rates_24h, width, label="24-hour horizon", color="#7fcdbb")

ax.set_ylabel("Positive label rate (%)")
ax.set_xticks(x)
ax.set_xticklabels([
    f"Training\n(Jul 2023 – Mar 2024)\nn = {stats['train']['n_rows']:,}",
    f"Test\n(Apr 2024 – Jul 2024)\nn = {stats['test']['n_rows']:,}"
])
ax.legend(loc="upper right")
ax.set_ylim(0, max(rates_24h) * 1.3)

for bars in [bars_4h, bars_24h]:
    for bar in bars:
        height = bar.get_height()
        ax.annotate(f"{height:.2f}%",
                    xy=(bar.get_x() + bar.get_width()/2, height),
                    xytext=(0, 3), textcoords="offset points",
                    ha="center", va="bottom", fontsize=9)

fig.text(0.5, 0.02, "Split boundary: 1 April 2024 (24-hour purge gap applied)",
         ha="center", fontsize=9, style="italic")

plt.tight_layout()
plt.subplots_adjust(bottom=0.15)
plt.savefig("/dbfs/tmp/thesis_fig_train_test_split.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: /dbfs/tmp/thesis_fig_train_test_split.png")

In [0]:
from pyspark.sql import functions as F

df = spark.read.table("hive_metastore.gold.gold_dataset")

# What % of transformers have ANY events on a typical day?
df.filter(F.to_date("DATE") == "2024-01-15").groupBy("ID_prefix").agg(
    F.sum("events_15m_cnt").alias("daily_events")
).select(
    F.count("*").alias("total_transformers"),
    F.sum(F.when(F.col("daily_events") > 0, 1).otherwise(0)).alias("with_events")
).show()

In [0]:
# =============================================================================
# CELL 7 — Figure 10: SCADA Event Counts (Panel A only)
# =============================================================================
# NOTE: This figure reads from gold_dataset. If events_15m_cnt is not in
# gold_dataset, you may need to read from gold_features (but use raw values,
# not scaled). Adjust SRC as needed.

df_events = spark.read.table(SRC_TABLE)

daily_events = (
    df_events.withColumn("date", F.to_date(TS_COL))
    .groupBy("date")
    .agg(
        F.sum("events_15m_cnt").alias("total_events"),
        F.mean(F.when(F.col("events_15m_cnt") > 0, 1).otherwise(0)).alias("pct_with_events"),
        F.count("*").alias("n_rows"),
    )
    .orderBy("date")
    .toPandas()
)
daily_events["date"] = pd.to_datetime(daily_events["date"])
daily_events = daily_events.set_index("date").sort_index()
daily_events["pct_with_events"] *= 100

fig, ax1 = plt.subplots(figsize=(12, 5), dpi=150)

# Panel A — Total event count
ax1.bar(daily_events.index, daily_events["total_events"], width=1, color="#2ca02c", alpha=0.7)
ax1.plot(daily_events.index, daily_events["total_events"].rolling(7, center=True).mean(),
         linewidth=2, color="#006400", label="7-day rolling mean")
ax1.set_ylabel("Total Events", fontsize=11)
ax1.legend(loc="upper right", fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_title("(a) Daily SCADA Event Count Across All Transformers", fontsize=11, fontweight="bold")

plt.tight_layout()
plt.savefig("/dbfs/tmp/thesis_fig10_events_panel_a.png", dpi=300, bbox_inches="tight")
plt.show()
print("Saved: /dbfs/tmp/thesis_fig10_events_panel_a.png")